In [ ]:
# Preparación (no modificar): ubica la raíz del repo, la usa como directorio de trabajo
# y pone codigo/ en el path. Funciona igual abriendo el notebook desde notebooks/ o desde
# la raíz, y también ejecutándolo con codigo/run_nb.py.
import os, sys
_raiz = os.getcwd()
while not os.path.isdir(os.path.join(_raiz, "codigo")) and os.path.dirname(_raiz) != _raiz:
    _raiz = os.path.dirname(_raiz)
os.chdir(_raiz)
sys.path.insert(0, os.path.join(_raiz, "codigo"))
print("Raíz del repo:", _raiz)

# Auditoría de los precios del estudio — réplica y actualización de Aradillas (2018)

**Qué hace este notebook.** Verifica, contra la fuente original de INEGI, los precios que
alimentan la réplica 2014 y la actualización 2022. Es un mecanismo de transparencia: cada
afirmación del reporte de hallazgos sobre precios debería poder comprobarse aquí, corriendo
el notebook.

**Por qué hace falta.** Los dos años se construyen igual que en el estudio original —un
precio de referencia por ciudad, deflactado con el INPC de esa ciudad hasta la ventana de
levantamiento de la ENIGH— pero **con bases distintas del INPC**:

| | precio de referencia | base del INPC | ventana |
|---|---|---|---|
| 2014 | junio 2011 | 2ª quincena de diciembre de 2010 | ago-nov 2014 |
| 2022 | julio 2018 | 2ª quincena de julio de 2018 | ago-nov 2022 |

Si el empalme entre bases estuviera mal, aparecería como un salto de nivel entre años y se
leería como cambio económico. Esa es la hipótesis que se pone a prueba.

**Método.** Comparar contra los **niveles observados** (pesos por kilo, litro o pieza) que
INEGI publica en su consulta de precios promedio del INPC. Los niveles son absolutos y no
dependen de la base, así que sirven para auditar el empalme.

**Qué NO cubre.** No audita los microdatos de la ENIGH, ni la estimación del sistema de
demanda, ni los Censos Económicos. Solo los precios.

---

### Dos fuentes con papeles opuestos

| carpeta | qué contiene | quién la usa |
|---|---|---|
| `Data_2022/data_inp_pp/` | precios de **referencia** (julio 2018), que el pipeline deflacta hasta la ventana | `datos_2022.precios_referencia` |
| `Data_auditoria/` | precios **observados** en la ventana (ago-nov de 2014 y 2022) | **solo este notebook** |

**El pipeline nunca debe leer `Data_auditoria/`.** Si lo hiciera, la auditoría estaría
validando los precios contra sí mismos: la correlación daría 1 por construcción y dejaría de
detectar cualquier problema. El valor de esta verificación está en que la fuente es
independiente del insumo.

Por la misma razón, los precios observados de la ventana **no sustituyen** a los de
referencia en el pipeline, aunque existan y sean más directos: eso se discute en la
sección 5.

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "codigo")

import numpy as np
import pandas as pd

import datos_2014, datos_2022

DATOS = "datos/Data_auditoria/"
panel14 = pd.read_csv(DATOS + "panel_precios_2014.csv")
panel22 = pd.read_csv(DATOS + "panel_precios_2022.csv")
cat10 = pd.read_csv(DATOS + "catalogo_base_2010.csv")
cat18 = pd.read_csv(DATOS + "catalogo_base_2018.csv")
desglose14 = pd.read_csv(DATOS + "panel_desglose_2014.csv")
desglose22 = pd.read_csv(DATOS + "panel_desglose_2022.csv")

print(f"panel 2014 (base 2010): {panel14['ciudad'].nunique()} ciudades, "
      f"{panel14['generico'].nunique()} genéricos")
print(f"panel 2022 (base 2018): {panel22['ciudad'].nunique()} ciudades, "
      f"{panel22['generico'].nunique()} genéricos")
print(f"catálogos: {len(cat10)} genéricos en base 2010, {len(cat18)} en base 2018")

## Cómo se obtuvieron estos datos

Los archivos de `Data_auditoria/` son agregados (mediana por ciudad, genérico y unidad) de
consultas a la app de precios promedio del INPC. Se guardan agregados porque las descargas
completas pesan unos 18 MB; los agregados bastan para reproducir todas las pruebas.

Para regenerarlos, la consulta es un POST a `Exportacion.aspx` precedido de una llamada a
`ObtieneCountReg`, que deja la selección en la sesión del servidor:

```python
# 1) abrir sesión (guardar cookies) en https://www.inegi.org.mx/app/preciospromedio/
# 2) POST a Servicios/ArbolAjaxInteraccion.asmx/ObtieneCountReg con JSON:
#    {"obsolet_series": ",001,006,...,",   # claves de genérico, 3 dígitos
#     "series2": "", "pi": "201408", "pf": "201411",
#     "entidades": ",01_01,01_02,...,",    # códigos de ciudad
#     "countreg": 1, "cab": ""}            # cab="" base 2010; cab="18" base 2018
# 3) POST a Exportacion.aspx con los mismos pi/pf/ent, tipo=CSV y bs igual a cab
```

**Dos trampas** que conviene anotar, porque no son obvias:

1. **Las claves de genérico cambian entre bases.** Pan de caja es `007` en la base 2010 y
   `009` en la base 2018. Cruzar por clave entre años produce comparaciones de productos
   distintos; hay que cruzar por nombre.
2. **El periodo se pide con `page: 0`.** Con `page: 1` la API responde "No existe
   información disponible" aunque los datos existan.

## 1. Las dos bases del INPC no se traslapan

Primera comprobación: si las bases no comparten ningún mes, no se pueden empalmar
directamente y hay que verificar el nivel por otra vía.

In [ ]:
for etiqueta, panel in (("base 2010 (usada para 2014)", panel14),
                        ("base 2018 (usada para 2022)", panel22)):
    print(f"{etiqueta}: {panel['ciudad'].nunique()} ciudades")

print("\nCobertura de cada base, según la propia API de INEGI:")
print("  base 2010 (cab=''):   2011/01 a 2018/06")
print("  base 2018 (cab='18'): 2018/08 a 2024/06")
print("  -> no hay ningún mes en común")

print("\nArchivos locales del proyecto:")
inpc14 = np.loadtxt("datos/Data_2014/inpc_46_ciudades.asc")
f = inpc14[:, 0]
print(f"  inpc_46_ciudades.asc (2014): {f.min():.2f} a {f.max():.2f}")
fe, _ = datos_2022._serie_inpc("datos/Data_2022/data_ciudades/inpc_acapulco.CSV")
ok = np.isfinite(fe)
print(f"  data_ciudades/ (2022):       {fe[ok].min():.2f} a {fe[ok].max():.2f}")
print("  -> ningún archivo local cubre los dos años: el empalme se audita con fuente externa")

### 1.1 Qué cambió entre bases

Además de las claves, INEGI redefinió la unidad de algunos genéricos. Eso sí rompería una
comparación entre años, porque el mismo bien cambiaría de precio sin que cambie nada real.

In [ ]:
c10 = cat10.assign(g=cat10["generico"].map(datos_2022._norm)).set_index("g")
c18 = cat18.assign(g=cat18["generico"].map(datos_2022._norm)).set_index("g")
comunes = c10.index.intersection(c18.index)

cambios = [(g, c10.loc[g, "unidad"], c18.loc[g, "unidad"]) for g in comunes
           if c10.loc[g, "unidad"] != c18.loc[g, "unidad"]]
print(f"genéricos en ambas bases: {len(comunes)} | con unidad distinta: {len(cambios)}\n")

usados = {datos_2022._norm(v) for v in datos_2022.GENERICO.values()}
print("Los que usa el estudio:")
for g, u10, u18 in sorted(cambios):
    if g in usados:
        print(f"  {g:<24} {u10:>8}  ->  {u18:>8}")

print("\nY las claves, para el mismo genérico:")
for g in ("pan de caja", "huevo", "pollo"):
    if g in comunes:
        print(f"  {g:<14} base 2010: {c10.loc[g, 'clave']:>4}   base 2018: {c18.loc[g, 'clave']:>4}")

**Resultado.** Dos genéricos del estudio cambian de definición entre bases: **huevo pasa de
docena a kilo** y **autobús foráneo de viaje a boleto**. Una docena pesa unos 0.7 kg, así que
el mismo huevo "sube" cerca de 40 % solo por el cambio de unidad.

Esto no invalida la comparación entre años —cada año deflacta dentro de su propia base, como
se verifica en la sección 3— pero sí obliga a no cruzar nunca por clave y a vigilar las
unidades, que es justo lo que destapó el error de la sección siguiente.

## 2. N17 — unidades mezcladas dentro del mismo genérico

Un genérico se cotiza en varias presentaciones. Si se toma la mediana de todas sin separar
por unidad, se promedian pesos por kilo con pesos por docena. El pipeline de 2022 hacía eso.

In [ ]:
import glob
ref = pd.concat([datos_2022._leer_inegi(f)
                 for f in sorted(glob.glob("datos/Data_2022/data_inp_pp/*.CSV"))],
                ignore_index=True)
ref["g"] = ref["Genérico"].map(datos_2022._norm)
ref["u"] = ref["Unidad"].astype(str).str.strip()
ref["p"] = pd.to_numeric(ref["Precio promedio"], errors="coerce")
ref = ref.dropna(subset=["p"])

sub = ref[ref["g"] == "huevo"]
print("Huevo — precios de referencia de julio 2018, por unidad:")
print(sub.groupby("u")["p"].agg(n="size", mediana="median").to_string())
print(f"\nmediana mezclando todo : {sub['p'].median():.2f}")
print(f"mediana solo en kilo   : {sub.loc[sub['u'] == 'KG', 'p'].median():.2f}")
print("La unidad del deflactor (base 2018) es el kilo, así que la mezcla subestima el precio.")
print("\nOjo con las presentaciones: PAQ mezcla paquetes de 18 y de 30 piezas.")
for e in sub.loc[sub["u"] == "PAQ", "Especificación"].astype(str).unique()[:4]:
    print("   ", e[:70])

In [ ]:
mezclados = []
for g, s in ref[ref["g"].isin(usados)].groupby("g"):
    if s["u"].nunique() > 1:
        modal = s["u"].mode().iloc[0]
        objetivo = datos_2022.UNIDAD_FORZADA.get(g, modal)
        f = s[s["u"] == objetivo]
        if len(f):
            mezclados.append((g, s["u"].nunique(), s["p"].median(), f["p"].median(),
                              f["p"].median() / s["p"].median()))
t = pd.DataFrame(mezclados, columns=["generico", "unidades", "mezclado", "filtrado", "razon"])
print(f"genéricos del estudio con más de una unidad: {len(t)} de {len(usados)}")
print(f"con sesgo mayor al 5 %: {(t['razon'].sub(1).abs() > 0.05).sum()}\n")
print(t.reindex(t["razon"].sub(1).abs().sort_values(ascending=False).index)
       .head(8).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

**Corrección aplicada.** `datos_2022.precios_referencia` ahora toma la **unidad modal del
propio archivo**, forzando `huevo → KG` porque ahí la modal (paquete) no es homogénea.

Dos detalles que costaron intentos fallidos y conviene dejar escritos:

- Usar la unidad del **catálogo** en vez de la modal produce disparates (nopales ×0.066,
  piña ×1.53), porque deja solo el 10-20 % de las filas.
- Filtrar a secas cuesta **cobertura**: el huevo quedaba en 31 de 46 ciudades, y las demás
  caerían al respaldo de "mediana nacional", que destruye la variación entre ciudades —el
  insumo que identifica el modelo—. Por eso, en las ciudades sin cotización en la unidad
  objetivo se usa su precio mezclado reescalado por la razón nacional.

In [ ]:
p, orig = datos_2022.precios_referencia("datos/Data_2022/", con_originales=True)
g_idx = p.index.get_level_values("generico")
faltan = [(x, int((g_idx == x).sum())) for x in usados if (g_idx == x).sum() not in (0, 46)]
print(f"genéricos sin las 46 ciudades tras la corrección: {faltan if faltan else 'ninguno'}")

h = p.xs("huevo", level="generico")
print(f"\nhuevo: {len(h)} ciudades | mediana {h.median():.2f} | "
      f"desviación estándar de ln P entre ciudades {np.log(h).std():.3f}")
print("(mezclando: 46 ciudades, mediana 27.88, std ln 0.230 — se corrige el nivel sin perder variación)")

## 3. ¿Los niveles de precio son correctos?

La prueba central: comparar el precio que produce cada pipeline contra el precio observado
que publica INEGI para la misma ventana, con unidades homogéneas en ambos lados.

In [ ]:
def unidad_canonica(panel, generico):
    """Unidad de referencia: la misma regla que usa el pipeline (N17).

    Modal por número de observaciones, salvo los genéricos donde la modal no es
    homogénea (huevo: la modal es PAQ, que mezcla paquetes de 18 y de 30 piezas).
    Aplicar aquí una regla distinta a la del pipeline haría que esta auditoría
    comparara cosas distintas — y es justo el error que documenta la sección 2.
    """
    s = panel[panel["generico"] == generico]
    if not len(s):
        return None
    forzada = datos_2022.UNIDAD_FORZADA.get(datos_2022._norm(generico))
    if forzada is not None and (s["unidad"] == forzada).any():
        return forzada
    return s.groupby("unidad")["n"].sum().idxmax()

def observado(panel, generico):
    """Precio observado por ciudad, en la unidad de referencia del genérico."""
    s = panel[(panel["generico"] == generico)
              & (panel["unidad"] == unidad_canonica(panel, generico))]
    return s.set_index("ciudad")["precio_mediano"]

# --- 2014: precios del pipeline (referencia jun-2011 deflactada) ---------------
P46, _, _ = datos_2014._precios_por_ciudad("datos/Data_2014/")
ref14 = np.loadtxt("datos/Data_2014/precios_promedio_46_ciudades_junio_2011.asc")
claves = [f"{int(e):02d}{int(m):03d}" for e, m in zip(ref14[:, 0], ref14[:, 1])]
fila = {c: i for i, c in enumerate(claves)}

PAR = {"tortillas": "Tortilla de maíz", "pan_blanco": "Pan blanco", "pollo_entero": "Pollo",
       "bistec_res": "Carne de res", "huevo": "Huevo",
       "leche_pasteurizada": "Leche pasteurizada y fresca", "jitomate": "Jitomate",
       "manzana": "Manzana", "refrescos_envasados": "Refrescos envasados",
       "autobus_foraneo": "Autobús foráneo", "transporte_aereo": "Transporte aéreo"}

filas = []
for prod, gen in PAR.items():
    med = observado(panel14, gen)
    a, b = [], []
    for ciudad, clave in datos_2014.CIUDAD_A_CLAVE.items():
        i = fila.get(clave)
        if i is not None and ciudad in med.index:
            a.append(P46[prod][i]); b.append(med[ciudad])
    a, b = np.array(a), np.array(b)
    filas.append((prod, np.median(a), np.median(b), np.median(b) / np.median(a),
                  np.corrcoef(a, b)[0, 1], len(a)))
t14 = pd.DataFrame(filas, columns=["producto", "nuestro", "INEGI", "razon_nivel",
                                   "corr_entre_ciudades", "ciudades"])
print("2014 — precio del pipeline vs observado (ago-nov 2014)")
print(t14.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

In [ ]:
# --- 2022: precios del pipeline (referencia jul-2018 deflactada) ---------------
fac = datos_2022.deflactar_precios("datos/Data_2022/", p,
                                   sorted({c for c, _ in p.index}), orig, verbose=False)
fac_global = float(np.median(list(fac.values())))

PAR22 = dict(PAR, pan_de_caja="Pan de caja")
filas = []
for prod, gen in PAR22.items():
    gnorm = datos_2022._norm(gen)
    med = observado(panel22, gen)
    a, b = [], []
    for ciudad_norm in {c for c, x in p.index if x == gnorm}:
        nombre = orig[ciudad_norm]
        if nombre not in med.index:
            continue
        a.append(p[(ciudad_norm, gnorm)] * fac.get((ciudad_norm, gnorm), fac_global))
        b.append(med[nombre])
    if len(a) < 5:
        continue
    a, b = np.array(a), np.array(b)
    filas.append((prod, np.median(a), np.median(b), np.median(b) / np.median(a),
                  np.corrcoef(a, b)[0, 1], len(a)))
t22 = pd.DataFrame(filas, columns=["producto", "nuestro", "INEGI", "razon_nivel",
                                   "corr_entre_ciudades", "ciudades"])
print("2022 — precio del pipeline vs observado (ago-nov 2022)")
print(t22.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

**Resultado 1: el empalme entre bases es correcto.** Los niveles cuadran con lo publicado —en
2014, la mayoría de los productos queda entre 0.94 y 1.10 del precio observado—. Como cada
año deflacta dentro de su propia base, el cambio de base **no** introduce un salto artificial
entre 2014 y 2022. Era la hipótesis principal a descartar, y queda descartada.

## 4. El hallazgo: los niveles son correctos, la variación entre ciudades no siempre

La columna `corr_entre_ciudades` de las tablas anteriores es la que importa para el modelo.
El sistema NEIO identifica el poder de mercado (`β_η`) con la **variación de precios entre
mercados**: si nuestro precio ordena las ciudades distinto a como lo hacen los precios reales,
el parámetro no mide lo que dice medir.

In [ ]:
comp = (t14[["producto", "corr_entre_ciudades"]].rename(columns={"corr_entre_ciudades": "2014"})
        .merge(t22[["producto", "corr_entre_ciudades"]].rename(columns={"corr_entre_ciudades": "2022"}),
               on="producto", how="outer"))
print("Correlación entre nuestro precio por ciudad y el observado")
print(comp.sort_values("2014").to_string(index=False, float_format=lambda x: f"{x:.3f}"))

In [ ]:
# ¿Se explica por mezclar productos heterogéneos dentro del genérico?
aereo = desglose14[desglose14["generico"] == "Transporte aéreo"]
print("Transporte aéreo 2014, por tipo de vuelo:")
print(aereo.groupby("tipo")["precio_mediano"].agg(n="size", mediana="median").to_string())

def observado_vuelos(solo_nacionales):
    s = aereo if not solo_nacionales else aereo[aereo["tipo"] != "internacional"]
    return s.groupby("ciudad").apply(
        lambda x: np.average(x["precio_mediano"], weights=x["n"]))

for excluir in (False, True):
    med = observado_vuelos(excluir)
    a, b = [], []
    for ciudad, clave in datos_2014.CIUDAD_A_CLAVE.items():
        i = fila.get(clave)
        if i is not None and ciudad in med.index:
            a.append(P46["transporte_aereo"][i]); b.append(med[ciudad])
    a, b = np.array(a), np.array(b)
    etq = "solo vuelos nacionales" if excluir else "todos los vuelos"
    print(f"  {etq:<24} razón {np.median(b)/np.median(a):.3f} | correlación {np.corrcoef(a, b)[0,1]:.3f}")

**Resultado 2: la variación transversal es ruido en algunas categorías.** En 2014, autobús
foráneo correlaciona 0.02 con los precios observados y refrescos −0.21.

**Y no se debe a mezclar productos.** Quitar los vuelos internacionales —que cuestan cinco
veces más que los nacionales— deja la correlación prácticamente igual. La causa es el método
mismo: arrastrar un precio de junio de 2011 durante tres años con el índice de cada ciudad no
reconstruye la estructura de precios entre ciudades. 2022 sale mejor porque su arrastre es más
corto (2018 a 2022).

**Consecuencia:** los `β_η` de **Transporte foráneo** y, en menor grado, **Bebidas** no son
interpretables como poder de mercado, por significativos que resulten. De hecho, con controles
de costo sectoriales, Transporte foráneo aparece significativo en 2022 (β 0.466, t 2.89): un
caso donde el estadístico dice una cosa y los datos dicen otra.

Esto es una crítica al **diseño del estudio original**, que construye así sus precios, no a la
implementación de la réplica.

## 5. Resumen y decisiones

| # | Verificación | Resultado |
|---|---|---|
| 1 | Empalme entre bases del INPC | ✅ correcto: cada año deflacta dentro de su base |
| 2 | Niveles de precio contra INEGI | ✅ razonables en ambos años |
| 3 | Unidades mezcladas (N17) | ⚠ error encontrado y **corregido** |
| 4 | Variación entre ciudades | ❌ no es real en transporte y bebidas |

**Decisiones registradas:**

- **Transporte aéreo no se filtra** por tipo de vuelo. En 2014 no es posible —el archivo de
  COFECE ya viene agregado por ciudad, sin especificaciones— y filtrar solo en 2022 rompería
  la comparabilidad entre años, que es el objetivo del proyecto. Además se verificó que
  filtrar no arregla el problema.
- **Los `β_η` de Transporte foráneo y Bebidas se reportan como no interpretables**, con esta
  auditoría como sustento.

**¿Y por qué no usar directamente los precios observados en el pipeline?** Existen para 57
de los 61 genéricos del estudio en ambas ventanas, así que sería viable. La razón para no
hacerlo es que **8 genéricos cambian de unidad entre las dos ventanas** (huevo de docena a
kilo; melón, piña, lechuga y nopales de kilo a pieza; autobús de viaje a boleto). Deflactar
con un índice —adimensional— preserva la unidad del precio base, de modo que el método
heredado es inmune a esos cambios de definición; usar observados obligaría a inventar
factores de conversión. La celda siguiente lo verifica.

**Limitaciones de esta auditoría.** El panel cubre 12 genéricos de los ~60 del estudio,
elegidos como representativos de cada categoría; las conclusiones sobre productos no incluidos
son por extensión, no por medición directa. Los precios observados se resumen con la mediana
sobre especificaciones y meses, igual que el pipeline, así que comparten ese criterio.

In [ ]:
n10 = {datos_2022._norm(x): u for x, u in zip(cat10["generico"], cat10["unidad"])}
n18 = {datos_2022._norm(x): u for x, u in zip(cat18["generico"], cat18["unidad"])}
usados = {datos_2022._norm(v) for v in datos_2022.GENERICO.values()}

disp = [g for g in usados if g in n10 and g in n18]
print(f"genéricos del estudio: {len(usados)} | con precio observado en AMBAS ventanas: {len(disp)}")
print(f"  sin dato en 2014: {sorted(g for g in usados if g not in n10)}")
print(f"  sin dato en 2022: {sorted(g for g in usados if g not in n18)}")

cambian = [(g, n10[g], n18[g]) for g in disp if n10[g] != n18[g]]
print(f"\ncambian de unidad entre ventanas: {len(cambian)}")
for g, a, b in sorted(cambian):
    print(f"  {g:<22} {a:>8}  ->  {b:>8}")
print("\nDeflactar preserva la unidad del precio base; usar observados exigiría convertirlas.")
